# Blues Lick Finder: RAG Baseline

A minimal RAG baseline over blues guitar lesson videos, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

Given a song or technique, it retrieves candidate lessons and asks the LLM to rank them by teaching quality.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Each document is one YouTube lesson video: metadata from a `yt-dlp` search (no API key needed) plus the English transcript from `youtube-transcript-api` when one exists.

The first run fetches from YouTube and caches everything to `data/lessons.json`. Later runs load from that file. Delete it to re-fetch.

In [2]:
from pathlib import Path

import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi

DATA_PATH = Path("../data/lessons.json")

TOPICS = [
    "BB King box blues lick lesson",
    "Freddie King Hideaway guitar lesson",
    "Stevie Ray Vaughan Pride and Joy guitar lesson",
    "blues turnaround licks lesson",
    "blues string bending technique lesson",
    "minor pentatonic blues licks beginner lesson",
    "Texas blues shuffle rhythm guitar lesson",
    "slow blues soloing lesson",
]
RESULTS_PER_TOPIC = 5

transcript_api = YouTubeTranscriptApi()


def fetch_transcript(video_id):
    try:
        transcript = transcript_api.fetch(video_id, languages=["en"])
    except Exception:
        return None
    return " ".join(s.text for s in transcript.snippets)


def fetch_lessons(topics, results_per_topic):
    lessons = {}
    ydl_opts = {"quiet": True, "no_warnings": True, "skip_download": True}

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for topic in topics:
            print(f"Searching: {topic}")
            result = ydl.extract_info(f"ytsearch{results_per_topic}:{topic}", download=False)

            for entry in result["entries"]:
                video_id = entry["id"]
                if video_id in lessons:
                    continue

                lessons[video_id] = {
                    "video_id": video_id,
                    "url": f"https://www.youtube.com/watch?v={video_id}",
                    "title": entry.get("title"),
                    "channel": entry.get("channel"),
                    "description": (entry.get("description") or "")[:1000],
                    "tags": " ".join(entry.get("tags") or []),
                    "duration_sec": entry.get("duration"),
                    "view_count": entry.get("view_count"),
                    "topic": topic,
                    "transcript": fetch_transcript(video_id),
                }

    return list(lessons.values())


if DATA_PATH.exists():
    documents = json.loads(DATA_PATH.read_text())
else:
    documents = fetch_lessons(TOPICS, RESULTS_PER_TOPIC)
    DATA_PATH.write_text(json.dumps(documents, indent=2))

with_transcript = sum(1 for d in documents if d["transcript"])
print(f"Loaded {len(documents)} lessons, {with_transcript} with transcripts")

Loaded 40 lessons, 27 with transcripts


## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [3]:
index = Index(
    text_fields=["title", "description", "tags", "transcript"],
    keyword_fields=["video_id", "channel", "topic"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=5)

In [5]:
instructions = """
You're a blues guitar teacher helping a student find the best lesson videos
for the song or technique in their QUERY. YouTube ranks videos by engagement;
your job is to rank them by teaching quality instead.

Use only the candidate lessons in the CONTEXT. Skip candidates that aren't
about the requested song or technique. Rank the rest from best to worst
using this rubric, judged mostly from the transcript:

- Breakdown: splits the lick into small, clearly explained pieces
- Demonstration: plays it slowly before playing it at full speed
- Specifics: names frets, strings, fingers, or points to tab/notation
- Why: explains the theory or feel (scale, chord tones, phrasing) behind it
- Focus: gets to the lesson quickly instead of long intros and sales pitches

If a candidate has no transcript, rank it from the title and description and
say that your confidence is low.

For each ranked lesson, return: rank, title, channel, URL, a score from 1 to 5,
a one-sentence justification that cites the rubric, and practice-log tags
(song, technique, key if mentioned). If none of the candidates fit the
QUERY, say so.
""".strip()

TRANSCRIPT_CHARS = 3000


def build_prompt(query, search_results):
    candidates = []
    for doc in search_results:
        candidate = {k: v for k, v in doc.items() if k != "transcript"}
        if doc["transcript"]:
            candidate["transcript_excerpt"] = doc["transcript"][:TRANSCRIPT_CHARS]
        else:
            candidate["transcript_excerpt"] = None
        candidates.append(candidate)

    search_result_json = json.dumps(candidates, indent=2)

    user_prompt = f"""
<QUERY>
{query}
</QUERY>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [8]:
print(rag("Which lessons teach the BB King box licks best?"))

Here are the ranked lessons that focus on teaching the BB King box licks:

### Rank 1
- **Title:** Learn 10 Classic Blues Licks in the BB King Box!
- **Channel:** swiftlessons
- **URL:** [Watch here](https://www.youtube.com/watch?v=i6Ho6aubraw)
- **Score:** 5
- **Justification:** This lesson thoroughly breaks down 10 licks with clear explanations, specific fingerings, and plays them slowly, providing excellent value in terms of learning the BB King box.
- **Practice Log Tags:** BB King, technique

### Rank 2
- **Title:** The BB King Box & Beyond - Blues Lead Guitar Lesson
- **Channel:** swiftlessons
- **URL:** [Watch here](https://www.youtube.com/watch?v=pZz-QHt8UBc)
- **Score:** 4
- **Justification:** This lesson effectively reviews the BB King box and introduces adjacent scales, providing logical structure and detail, although some sections could be more concise.
- **Practice Log Tags:** BB King, technique

### Rank 3
- **Title:** The Ultimate B.B. King Lick Tutorial
- **Channel:** M